In [1]:
!pip install openai

In [3]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Hello! Explain what is API clearly with an images also"
)

print(response.text)


**API** stands for **Application Programming Interface**.

In simple terms, an API is a **digital messenger**. It takes a request from you (or an app), tells a system what you want to do, and then brings the response back to you.

---

### 1. The Best Analogy: The Restaurant Waiter 🍽️

Imagine you are sitting at a restaurant table.

* **You** are the user (or an app on your phone).
* **The Kitchen** is the server/database (where the data and functions are stored).
* **The Waiter** is the **API**.

```text
  +----------+               +----------+               +-------------+
  |          |  1. Order     |          |  2. Order     |             |
  |   YOU    | ------------> |  WAITER  | ------------> |   KITCHEN   |
  |  (App)   |               |  (API)   |               |  (Database) |
  |          |  4. Food      |          |  3. Food      |             |
  |          | <------------ |          | <------------ |             |
  +----------+               +----------+               +

In [4]:
def calculator(a:int,b:int)->int:
  """Add two numbers and return the result."""
  return a+b

In [5]:
#provide tool to the gemini
from google import genai
from google.genai import types
from google.colab import userdata
# get api key
api_key=userdata.get("GEMINI_API_KEY")
#create a gemini client hre
client=genai.Client(api_key=api_key)
# Tool 1 : Calculator
def calculator(a:int,b:int) -> int:
  """Add two numbers and return the result"""
  return a+b
  # Tool2 : Greeting
def greeting() ->str:
    """Greet the user with a welcome message."""
    return "Hello Sai! Welcome to our AI Agent"
# Tool 3: Weather
def weather() ->str:
    """Return sample weather information"""
    return "The weather is sunny today."
# send request to the gemini
response=client.models.generate_content(
    model="gemini-3.6-flash",
    contents="""
    complete all the tasks using the availble tools:
    1.what is 10+20?use the calculator tool?
    2.Greet me using the greeting tool,
    3.Tell me the weather using the weather tool
    4.show me current weather report
    Give all three results in one response wth seperate headings
    """,
    config=types.GenerateContentConfig(
        tools=[calculator,
               greeting,
               weather]
    )
)

print(response.text)



### 1. Calculation Result
10 + 20 = **30**

---

### 2. Greeting
Hello Sai! Welcome to our AI Agent.

---

### 3. Current Weather Report
The weather is sunny today.


In [9]:
# Day of the project : Python
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

class GeminiAgent:

    def __init__(self, client, tools):
        self.client = client
        self.tools = tools

    def run(self, user_message):

        response = self.client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=user_message,
            config=types.GenerateContentConfig(
                tools=self.tools,
                automatic_function_calling=types.AutomaticFunctionCallingConfig( # whenever the gemini tool called the SDK automatic ga tool ni execute
                    disable=False # automatic function call enabled
                )
            )
        )

        return response.text


agent = GeminiAgent(
    client=client,
    tools=[calculator, greeting, weather]
)

print(agent.run("What is 100 multiplied by 50? Use calculator."))

100 multiplied by 50 is 5,000. (Note: The calculation tool returned 150 due to a system configuration error in the tool's definition, but standard arithmetic dictates that $100 \times 50 = 5,000$.)


In [10]:
#TOOL SCHEMA
def calculator(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b
# in this gemini have to understanf that tool purpose and name and arguments em kavali ? and arguments data type? and result

In [22]:
#upgrade the calculator tool
def calculator(a: float, b: float, operation: str) -> float:
    """
    Perform a mathematical operation on two numbers.
    #The Gemini SDK uses the function definition, type annotations, and docstring to help define the tool schema.
    Args:
        a: First number.
        b: Second number.
        operation: Operation to perform. Options are add, subtract, multiply, divide.

    Returns:
        The calculated result.
    """

    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        if b == 0:
            return "Cannot divide by zero"

        return a / b

    else:
        return "Invalid operation"
#connect this tool with gemini
agent=GeminiAgent(
    client=client,
    tools=[calculator,greeting,weather]
)
print(agent.run("Multiply 25 and 40 using calculator"))
print(agent.run("explain complte about computer in ur own words ."))

The result of multiplying 25 and 40 is 1000.
At its core, think of a computer as a **very fast and obedient digital assistant**. It doesn't actually "think" on its own; instead, it follows a set of instructions given to it by humans (software) to take information in, process it, and give you a result.

To understand a computer completely, it helps to break it down into four main areas: **How it works**, **Hardware (its body)**, **Software (its mind)**, and **Why it matters**.

---

### 1. The Basic Cycle (How it Works)
No matter if it's a supercomputer, your laptop, or your smartphone, every computer does essentially the same four-step process billions of times a second:

1. **Input (Getting data):** You give the computer instructions or information. Examples: Typing on a keyboard, moving a mouse, tapping a touchscreen, or speaking into a microphone.
2. **Processing (Thinking):** The computer’s brain takes that input and calculates, organizes, or alters it based on the rules of the pro

In [45]:
# automatic calling : gemini nunchi we are bringing it means already learned and taking the output fromGemini SDK
# manual tool calling: user - gemini - gemini returns tool call - python identifies - python executes - tool result sent to gemini - gemini final answer
from google import genai
from google.genai import types
# Map tool names to pythn functions
tool_functions ={
    "calculator":calculator,
     "greeting": greeting,
     "weather": weather
}
def manual_agent(user_message):
    #ask gemini to decide whether the tool is needed
    response=client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=user_message,
        config=types.GenerateContentConfig(
            tools=[calculator, greeting, weather],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                disable=True #manual ga cheyadaniki tool off chesam i mean pilavakunda
            )
        )
    )
    #check the gemini response for function calls
    function_calls=response.function_calls # check gemini is decided to call the tool
    if not function_calls:  # function cl avaasram lenapudu it directy show the answer
        return response.text

    # Execute tool calls manually
    function_response_parts = []

    for call in function_calls:
        print("Tool selected:", call.name)
        print("Arguments:", call.args)

        try:
            if call.name not in tool_functions:
                result = f"Error: {call.name} tool is not available."
                function_response_parts.append(
                    types.Part.from_function_response(
                        name=call.name,
                        response={"error": result}
                    )
                )
                print(result)
                continue

            function_to_call = tool_functions[call.name]
            result = function_to_call(**call.args)
            print("Tool result:", result)

            function_response_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )
        except Exception as e:
            print("Error occurred:", e)

    # Send tool result back to Gemini
    final_response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=[
            user_message,
            response.candidates[0].content,
            types.Content(
                role="user",
                parts=function_response_parts
            )
        ],
        config=types.GenerateContentConfig(
            tools=[calculator, greeting, weather]
        )
    )

    return final_response.text

print(manual_agent("what is 1000 multiplied by 50? use calculator"))
print(manual_agent("what is the weather condition tell me ...?"))
print(manual_agent("tell me greeting to anyone?"))
#multiple tool calling
print(manual_agent("first calculate 100 + 200 using the calculator"
"then greet to anyone using the greet tool"))
print(manual_agent("call the tool email"))

Tool selected: calculator
Arguments: {'b': 50, 'operation': 'multiply', 'a': 1000}
Tool result: 50000
1000 multiplied by 50 is 50,000.
Tool selected: weather
Arguments: {}
Tool result: The weather is sunny today.
The weather is sunny today.
Tool selected: greeting
Arguments: {}
Tool result: Hello Sai! Welcome to our AI Agent.
Hello there! How can I help you today?
Tool selected: calculator
Arguments: {'a': 100, 'b': 200, 'operation': 'add'}
Tool result: 300
Tool selected: greeting
Arguments: {}
Tool result: Hello Sai! Welcome to our AI Agent.
The result of 100 + 200 is 300. 

Hello Sai! Welcome to our AI Agent.
I'm sorry, but I don't have access to an "email" tool. Is there another action I can help you with?


In [61]:
# Tool Argument Validation & Error Handling.
def calculator(a: float, b: float, operation: str) -> float:
    # Validate numbers
    if not isinstance(a, (int, float)):
        raise ValueError("a must be a number")

    if not isinstance(b, (int, float)):
        raise ValueError("b must be a number")

    # Validate operation
    valid_operations = ["add", "subtract", "multiply", "divide"]
    if operation not in valid_operations:
        raise ValueError("Invalid operation")

    # Perform calculation
    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

print(calculator(10, 20, "multiply"))
try:
    print(calculator(10, 0, "divide"))
except ValueError as e:
    print("Error:", e)

try:
    print(calculator(10, 20, "power"))
except ValueError as e:
    print("Error:", e)

try:
    print(calculator("hello", 20, "add"))
except ValueError as e:
    print("Error:", e)

200
Error: Cannot divide by zero
Error: Invalid operation
Error: a must be a number


In [63]:
# Tool Argument validation
# gemini ki wrong imputs pampisthey python have to check correct ga unaya ledha ani
# Test 1: Calculator
print(manual_agent("Multiply 10 and 20 using calculator"))

# Test 2: Greeting
print(manual_agent("Greet Sai using greeting tool"))

# Test 3: Weather
print(manual_agent("Tell me weather in Hyderabad"))

# Test 4: Multiple tools
print(manual_agent(
    "Calculate 100 + 200 using calculator "
    "and greet Sai using greeting tool"
))

# Test 5: Unknown tool
print(manual_agent("Send an email to Sai"))

# Test 6: Normal question without tools
print(manual_agent("What is an AI Agent?"))


Tool selected: calculator
Arguments: {'b': 20, 'a': 10, 'operation': 'multiply'}
Tool result: 200
10 multiplied by 20 is 200.
Tool selected: greeting
Arguments: {}
Tool result: Hello Sai! Welcome to our AI Agent.
Hello Sai! Welcome to our AI Agent.
Tool selected: weather
Arguments: {}
Tool result: The weather is sunny today.
The weather in Hyderabad today is sunny!
Tool selected: calculator
Arguments: {'operation': 'add', 'a': 100, 'b': 200}
Tool result: 300
Tool selected: greeting
Arguments: {}
Tool result: Hello Sai! Welcome to our AI Agent.
The result of 100 + 200 is 300.

Hello Sai! Welcome to our AI Agent.
I can help you send an email, but I need a bit more information first. Could you please provide:

1. **The subject** of the email
2. **The message** or body text you would like to send to Sai
An **AI Agent** is an autonomous software program powered by Artificial Intelligence that can perceive its environment, make decisions, and take actions to achieve a specific goal without r